# Joint minutes → points calibration

Offline coupling-only training. This notebook **calls** `train_joint_calibration`; it does not reimplement OOF, overlay, $\beta$, or $\epsilon$ pools.

Production minutes-mean, minutes-distribution, and points-mean artifacts are fingerprinted and left unchanged. `2025-26` is closed: those parquets are never loaded here.

$g$ itself may be negative. Only $\hat\mu = \max(0, \hat P + g)$ is clipped. Final $\beta$ is fit on $v - g_{\mathrm{final}}$; $\epsilon$ pools stay cross-fitted from `coupling_oof_eligible` rows. The first base OOF fold is `coupling_warmup`.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path("/Users/alexgonzalez/Documents/nba_quant")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.points import add_points_features
from src.models.xgboost_models.joint_calibration import (
    HOLDOUT_SEASON,
    JOINT_CALIBRATION_ARTIFACT,
    assert_preholdout,
    evaluate_overlay_gate,
    overlay_gate,
    residual_metrics,
    save_joint_calibration,
    train_joint_calibration,
)
from src.models.xgboost_models.minutes import MINUTES_DISTRIBUTION_ARTIFACT

MINUTES_MEAN_ARTIFACT = (
    ROOT / "artifacts" / "models" / "minutes" / "xgboost_minutes.joblib"
)
POINTS_MEAN_ARTIFACT = (
    ROOT / "artifacts" / "models" / "points" / "xgboost_points.joblib"
)
PREHOLDOUT_SEASONS = [
    "2019-20",
    "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
]


In [2]:
frames = []
for season in PREHOLDOUT_SEASONS:
    path = (
        ROOT
        / "data"
        / "silver"
        / "nba"
        / season
        / "regular_season"
        / "player_gamelogs.parquet"
    )
    frames.append(pd.read_parquet(path))

df = pd.concat(frames, ignore_index=True)
assert HOLDOUT_SEASON not in set(df["season_year"].astype("string"))
assert_preholdout(df)

df = add_points_features(df)
appearances = df.loc[df["minutes"].gt(0)].copy()
appearances["game_date"] = pd.to_datetime(appearances["game_date"])
assert_preholdout(appearances)

print(f"Preholdout appearances: {len(appearances):,}")
print("Seasons:", sorted(appearances["season_year"].astype("string").unique()))


Preholdout appearances: 150,062
Seasons: ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


In [3]:
artifact, panel = train_joint_calibration(
    appearances,
    minutes_mean_path=MINUTES_MEAN_ARTIFACT,
    minutes_dist_path=MINUTES_DISTRIBUTION_ARTIFACT,
    points_path=POINTS_MEAN_ARTIFACT,
    return_panel=True,
)
saved = save_joint_calibration(artifact, JOINT_CALIBRATION_ARTIFACT)
print(f"Saved {saved}")


Saved /Users/alexgonzalez/Documents/nba_quant/artifacts/models/points/joint_calibration.joblib


In [4]:
print("base_oof_max_training_date:", artifact.base_oof_max_training_date)
print("final_calibration_max_date:", artifact.final_calibration_max_date)
print(f"n_preholdout_appearances: {artifact.n_preholdout_appearances:,}")
print(f"n_base_oof_eligible: {artifact.n_base_oof_eligible:,}")
print(f"n_coupling_oof_eligible: {artifact.n_coupling_oof_eligible:,}")
print(f"oof_coverage: {artifact.oof_coverage:.1%}")
print("epsilon_source:", artifact.epsilon_source)
print()
print("Fingerprints")
for name, payload in artifact.fingerprints.items():
    if isinstance(payload, dict):
        print(
            f"  {name}: trees={payload.get('n_estimators')} "
            f"cutoff={payload.get('training_cutoff')} "
            f"hash={payload.get('content_hash')}"
        )
    else:
        print(f"  {name}: {payload}")
print()
print("Exclusions")
for reason, count in sorted(artifact.exclusions.items()):
    print(f"  {reason}: {count:,}")
print()
print("Folds")
pd.DataFrame(artifact.fold_records)


base_oof_max_training_date: 2024-03-28 00:00:00
final_calibration_max_date: 2025-04-13 00:00:00
n_preholdout_appearances: 150,062
n_base_oof_eligible: 140,780
n_coupling_oof_eligible: 114,083
oof_coverage: 76.0%
epsilon_source: cross_fitted

Fingerprints
  minutes_mean: trees=750 cutoff=2024-03-18 00:00:00 hash=720734b2aeea19bf10e7d2a45a98d34438ea205c45fc05d6e240e543f57e6ef5
  minutes_distribution: trees=750 cutoff=2024-03-18 00:00:00 hash=97e51f18f666af556b4624df120743060c4e937fdda0dfc4e8db62f4771fcab0
  points_mean: trees=592 cutoff=2024-03-18 00:00:00 hash=2e758867e027f9689a3a4548a185d7112cb2275176c6ecc33c81f6803f1b7fe7

Exclusions
  coupling_warmup: 26,697
  warmup: 9,282

Folds


,fold,coupling_warmup,n_train,n_valid,valid_min_date,valid_max_date
0,1,True,9282,26697,2019-12-22 00:00:00,2021-03-23 00:00:00
1,2,False,35979,28619,2021-03-24 00:00:00,2022-02-25 00:00:00
2,3,False,64598,27754,2022-02-26 00:00:00,2023-03-08 00:00:00
3,4,False,92352,28394,2023-03-09 00:00:00,2024-03-28 00:00:00
4,5,False,120746,29316,2024-03-29 00:00:00,2025-04-13 00:00:00


In [5]:
bins = np.asarray(artifact.minutes_bins, dtype=float)
beta_rows = []
for bin_id, slope in sorted(artifact.beta.beta_by_bin.items()):
    lo = bins[bin_id]
    hi = bins[bin_id + 1]
    beta_rows.append(
        {
            "bin": bin_id,
            "minutes": f"[{lo:g}, {hi:g})",
            "beta": slope,
        }
    )
beta_table = pd.DataFrame(beta_rows)
print(f"beta_global: {artifact.beta.beta_global:.4f}")
print(f"shrinkage k: {artifact.beta.shrinkage:g}")
print(beta_table.round(4).to_string(index=False))

overlay = artifact.overlay
overlay_table = pd.DataFrame(
    {
        "feature": list(overlay.feature_names),
        "coef": overlay.coef,
    }
)
print(f"overlay intercept: {overlay.intercept:.4f}")
print("minutes_shock coefficient is constrained >= 0; g itself may be negative.")
print(overlay_table.round(4).to_string(index=False))
overlay_table.round(4)


beta_global: 0.5048
shrinkage k: 20
 bin  minutes   beta
   0  [0, 12) 0.4065
   1 [12, 18) 0.4494
   2 [18, 24) 0.4915
   3 [24, 30) 0.5328
   4 [30, 36) 0.6421
   5 [36, 64) 0.7462
overlay intercept: 0.0285
minutes_shock coefficient is constrained >= 0; g itself may be negative.
       feature    coef
 minutes_shock  0.1464
pts_per_min_10  0.0504
  usg_wmean_10 -0.0283
 start_rate_10  0.0270


,feature,coef
0,minutes_shock,0.1464
1,pts_per_min_10,0.0504
2,usg_wmean_10,-0.0283
3,start_rate_10,0.0270


In [6]:
eps = artifact.epsilon
edges = np.asarray(eps.bins, dtype=float)
pool_rows = []
for bin_id in sorted(eps.pools):
    hi = edges[bin_id + 1]
    hi_label = "inf" if np.isinf(hi) else f"{hi:g}"
    pool_rows.append(
        {
            "bin": bin_id,
            "mu_hat": f"[{edges[bin_id]:g}, {hi_label})",
            "n": int(len(eps.pools[bin_id])),
            "removed_mean": eps.removed_means[bin_id],
            "raw_std": eps.raw_stds[bin_id],
        }
    )
print(f"centered: {eps.centered}")
print(f"winsor: {eps.winsor}")
print(f"min_pool_size: {eps.min_pool_size}")
pd.DataFrame(pool_rows).round(4)


centered: True
winsor: (0.005, 0.995)
min_pool_size: 40


,bin,mu_hat,n,removed_mean,raw_std
0,0,"[0, 8)",48251,0.0377,3.4799
1,1,"[8, 14)",36517,-0.0562,4.9349
2,2,"[14, 20)",16407,-0.1537,5.9991
3,3,"[20, 28)",11100,-0.3265,6.8481
4,4,"[28, inf)",1808,-0.2899,7.4126


In [7]:
eligible = panel.loc[panel["coupling_oof_eligible"]].copy()
start_text = eligible["start_position"].astype("string").str.strip()
eligible["actual_starter"] = (
    start_text.notna()
    & start_text.ne("")
    & start_text.ne("nan")
    & start_text.ne("<NA>")
)
eligible["has_expected_role"] = eligible["start_rate_10"].notna()
eligible["expected_starter"] = eligible["start_rate_10"] >= 0.5
eligible["role_shock"] = (
    eligible["has_expected_role"]
    & (eligible["expected_starter"] != eligible["actual_starter"])
)

pts = eligible["pts"].to_numpy(dtype=float)
off_mu = eligible["points_hat"].to_numpy(dtype=float)
on_mu = np.maximum(0.0, off_mu + eligible["g_hat"].to_numpy(dtype=float))

slices = {
    "overall": np.ones(len(eligible), dtype=bool),
    "expected_starter": (
        eligible["has_expected_role"] & eligible["expected_starter"]
    ),
    "expected_bench": (
        eligible["has_expected_role"] & ~eligible["expected_starter"]
    ),
    "actual_starter": eligible["actual_starter"],
    "actual_bench": ~eligible["actual_starter"],
    "role_shock": eligible["role_shock"],
    "expected_starter_sat": (
        eligible["has_expected_role"]
        & eligible["expected_starter"]
        & ~eligible["actual_starter"]
    ),
    "expected_bench_started": (
        eligible["has_expected_role"]
        & ~eligible["expected_starter"]
        & eligible["actual_starter"]
    ),
}

rows = []
pairs = []
for name, mask in slices.items():
    mask = np.asarray(mask, dtype=bool)
    on = residual_metrics(pts[mask], on_mu[mask])
    off = residual_metrics(pts[mask], off_mu[mask])
    pairs.append((on, off))
    passed = overlay_gate(on, off)
    if passed is None:
        status = "skipped"
    else:
        status = "pass" if passed else "fail"
    rows.append(
        {
            "slice": name,
            "n": on["n"],
            "mae_on": on["mae"],
            "mae_off": off["mae"],
            "bias_on": on["bias"],
            "bias_off": off["bias"],
            "width_80_on": on["width_80"],
            "width_80_off": off["width_80"],
            "gate": status,
        }
    )

gate_table = pd.DataFrame(rows)
print(
    "Overlay-on vs overlay-off uses cross-fitted g on coupling_oof_eligible."
)
print(
    "Gate: n>=200, MAE_on <= MAE_off + 0.05, |bias_on| <= |bias_off| + 0.05."
)
print("evaluate_overlay_gate:", evaluate_overlay_gate(pairs))
gate_table.round(3)


Overlay-on vs overlay-off uses cross-fitted g on coupling_oof_eligible.
Gate: n>=200, MAE_on <= MAE_off + 0.05, |bias_on| <= |bias_off| + 0.05.
evaluate_overlay_gate: True


,slice,n,mae_on,mae_off,bias_on,bias_off,width_80_on,width_80_off,gate
0,overall,114083,4.525,4.523,-0.032,0.025,14.252,14.249,pass
1,expected_starter,52205,5.269,5.269,-0.063,0.016,16.673,16.660,pass
2,expected_bench,61428,3.902,3.900,-0.003,0.035,12.260,12.240,pass
3,actual_starter,53600,5.356,5.355,0.436,0.550,17.031,17.024,pass
4,actual_bench,60483,3.788,3.787,-0.447,-0.441,11.720,11.705,pass
5,role_shock,13691,4.843,4.874,0.826,0.838,15.511,15.574,pass
6,expected_starter_sat,6159,4.421,4.461,-1.238,-1.385,13.128,13.081,pass
7,expected_bench_started,7532,5.187,5.211,2.513,2.656,15.827,15.789,pass
